# 1.4 Chat Models & Embeddings

AI Agent Engineering · LogicStack — practice notebook

Run the cells in order. Change the code, break things, fix them. That is the practice.

Lesson page: https://logicstack.org/courses/ai-agent-engineering

In [ ]:
# Run this first: installs the exact versions this course uses.
!pip install -q "langchain==1.4.7" "langchain-core==1.6.13" "langchain-openai==1.7.2" "langgraph==1.2.16"

## Your API key

Some cells in this notebook call a model and need your own OpenAI API key. Paste it when asked. It is stored only in this notebook session, never saved to the file.

In [ ]:
import os, getpass
os.environ["OPENAI_API_KEY"] = getpass.getpass("Paste your OpenAI API key (input stays hidden): ")

## Learning objectives

- Initialise a chat model with `ChatOpenAI`, set `model`, `temperature`, and `max_tokens`, and explain what each knob does.
- Use `init_chat_model` as the provider-agnostic alternative, and swap providers (OpenAI, Anthropic, Ollama) without rewriting code.
- Create embeddings with `OpenAIEmbeddings`, compare `text-embedding-3-small` and `text-embedding-3-large` on dimensions, cost, and quality, and shrink vectors with the `dimensions` parameter.
- Embed a batch of texts efficiently with `embed_documents` and explain when `embed_query` is the wrong choice.
- Make a defensible model choice by weighing cost, quality, and latency, and note where data residency matters for EU users.

## Chat models: the engine of every chain

Every chain you have built so far ends at a chat model. A chat model is the one component that actually calls an LLM API. Everything else (prompts, parsers, retrievers) is plumbing around it. So it pays to understand the few parameters that matter, and how to switch models without rewriting your code.

Start with the workhorse of this course:

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  # reads OPENAI_API_KEY from a .env file

model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=500,
)

print(model.invoke("Name three European capitals and one fact about each.").content)

The three parameters worth knowing by heart:

- **`model`** picks which model you talk to. `gpt-4o-mini` is the sensible default for this course: cheap, fast, and good enough for most agent plumbing. When you need stronger reasoning (planning steps, writing code), step up to `gpt-4o` or a reasoning model.
- **`temperature`** controls randomness. `0` makes the model deterministic and to the point: best for classification, extraction, routing, and tool calls, where you want the same answer every time. Around `0.7` to `1.0` adds variety: better for brainstorming and creative writing. For agents, default to `0` unless you have a reason not to. A flaky tool-calling agent is often just a high-temperature agent.
- **`max_tokens`** caps the length of the response. It is a budget control: long replies cost more and take longer. Note that it limits *output* only; it does nothing about a too-long input.

One more that is easy to overlook: `timeout` (seconds) for how long the call may take before it gives up. In Module 6, `timeout` plus retries is your basic reliability story.

## The provider-agnostic way: `init_chat_model`

Hard-coding `ChatOpenAI` everywhere means every model swap touches your imports. LangChain 1.x prefers a factory function that resolves the right class from a string:

In [ ]:
from langchain.chat_models import init_chat_model

gpt   = init_chat_model("openai:gpt-4o-mini", temperature=0)
claude = init_chat_model("anthropic:claude-sonnet-4-5", temperature=0)
local = init_chat_model("ollama:llama3.1", temperature=0)

The `"provider:model"` prefix picks the integration; extra keyword arguments (`temperature`, `max_tokens`, `timeout`, `base_url`) are forwarded to the underlying class. Your chain code stays identical whichever model sits underneath, which is exactly what you want when you benchmark OpenAI against Anthropic, or swap a hosted model for a local one in tests.

The course uses explicit classes (`ChatOpenAI`, `ChatOllama`) so you always see which provider you are talking to. Both forms are fine; pick one per project and be consistent.

## Other chat models and the Ollama path

Anthropic's models are the common alternative to OpenAI, especially for long-context reasoning. The pattern is the same, just a different class and key:

In [ ]:
# pip install langchain-anthropic, set ANTHROPIC_API_KEY
from langchain_anthropic import ChatAnthropic

claude = ChatAnthropic(model="claude-sonnet-4-5", temperature=0)
print(claude.invoke("Summarise the GDPR in two sentences.").content)

Ollama runs models locally on your own machine. No API key, no per-token cost, and no data leaves your machine, which is a big deal for the EU angle below:

In [ ]:
# pip install langchain-ollama, then: ollama pull llama3.1
from langchain_ollama import ChatOllama

local = ChatOllama(
    model="llama3.1",
    base_url="http://localhost:11434",  # default; set it explicitly if your server lives elsewhere
    temperature=0,
)
print(local.invoke("What is an agent in two sentences?").content)

The knobs are the same as `ChatOpenAI` (plus Ollama-specific ones like `num_ctx` for the context window and `num_predict` for max output tokens). Drop a `ChatOllama` instance anywhere a `ChatOpenAI` would go and the chain does not care, which is why we write provider-agnostic chain code.

### A word on data residency (EU angle)

When you call a hosted model, your prompts travel to the provider's servers. For EU users and EU customer data this matters. OpenAI offers EU data residency on eligible plans (requests processed and stored in the EU); Anthropic has EU data processing options too. If your agent handles personal data and you cannot justify the transfer, Ollama keeps everything on your own machine, which sidesteps the question entirely. You do not need to solve GDPR in lesson 1.4, but you should know which of your tools sends data where. Module 6.5 covers the full picture.

## Embeddings: turning text into numbers

An embedding is a list of floats that represents the meaning of a piece of text. Similar meanings produce similar vectors, which is what lets a vector store answer "which documents are like this question?" in Module 3. For now, you need the mechanics: creating them, and what the settings change.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embedder = OpenAIEmbeddings(model="text-embedding-3-small")

one = embedder.embed_query("What is the capital of France?")
print(len(one))   # 1536
print(one[:5])    # [0.0213, -0.0441, ...] (the numbers mean nothing to a human)

### Small vs large: the only comparison that matters

OpenAI's current embedding lineup is two models. Pick between them with your eyes open:

| | text-embedding-3-small | text-embedding-3-large |
|---|---|---|
| Dimensions | 1,536 | 3,072 |
| Cost per 1M tokens | about $0.02 | about $0.13 |
| Max input | 8,191 tokens | 8,191 tokens |
| Quality (MTEB) | good, roughly 62 | better, roughly 65 |

Small is the default recommendation. It costs roughly six times less and the quality gap is small on most real workloads. Step up to large when retrieval quality is the bottleneck and you have measured that it helps, not before. (Prices drift; check the current OpenAI pricing page before budgeting a big job.)

### Shrinking vectors with `dimensions`

Both models support Matryoshka-style truncation: you can ask for fewer dimensions and the vectors stay useful. Fewer dimensions mean less storage and faster search:

In [ ]:
compact = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=512,   # truncate from 3072 to 512; quality loss is usually 1–2 points
)

This is the single most underused optimisation in RAG. A large model truncated to 512 dimensions is often cheaper to store and search than the small model at full size, while beating it on quality.

### Batches: `embed_documents`, not `embed_query` in a loop

Embedding is billed per token and rate-limited per request, so how you batch matters. `embed_documents` sends many texts in as few requests as possible:

In [ ]:
texts = [
    "The GDPR requires a lawful basis for processing personal data.",
    "The EU AI Act bans certain high-risk AI practices.",
    "LangChain chains are built from runnables.",
]

vectors = embedder.embed_documents(texts)   # one batched call
print(len(vectors), len(vectors[0]))        # 3 1536

`embed_query` is for a single search query (one text, one vector). Looping `embed_query` over ten thousand documents is ten thousand requests; `embed_documents` over the same list is a handful. The `chunk_size` parameter controls how many texts go into each request (default 1000). If you hit rate limits on a big corpus, lower it.

### The local alternative

For offline work, Ollama ships embedding models too:

In [ ]:
# ollama pull nomic-embed-text
from langchain_ollama import OllamaEmbeddings

local_embed = OllamaEmbeddings(model="nomic-embed-text")
vec = local_embed.embed_query("What is an agent?")
print(len(vec))  # 768

768 dimensions, zero API cost, and the text never leaves your machine. Quality trails the OpenAI models, but for development, tests, and small corpora it is more than fine.

## Picking a model: cost, quality, latency

You will pick models dozens of times in this course. Use this framework instead of vibes:

1. **Start cheap and fast.** `gpt-4o-mini` for chat, `text-embedding-3-small` for embeddings. Build the whole pipeline first.
2. **Measure before upgrading.** If the agent gives wrong answers, check whether the failure is the model or your prompt/retrieval. Upgrading the model hides a bad prompt for about a week.
3. **Spend where it counts.** In a pipeline, put the strong model only on the step that needs judgement (planning, final synthesis) and the cheap model on the mechanical steps (classification, formatting). `init_chat_model` makes this a configuration change, not a rewrite.
4. **Remember latency.** Local Ollama models have no API cost but answer slowly on a laptop CPU. A hosted small model often beats a local big one on both speed and quality, unless your constraints forbid sending data out.
5. **Watch the context window.** Every model has a maximum input size. Stuffing history plus documents plus tools into a small window fails loudly (lesson 1.5 is entirely about this).

## What can go wrong

- **Wrong model name.** `model="gpt-4o-min"` or a model your key cannot access raises an API error that looks scarier than it is. Copy the name from the provider's current model list; names change and old tutorials go stale.
- **Context-window overflow.** Too much history plus too many retrieved documents in one prompt and the API rejects the call (or, worse, silently truncates). This is the signature failure of chat agents and the subject of lesson 1.5.
- **High temperature in agents.** Temperature 0.8 on a tool-calling agent produces creative tool arguments and non-deterministic routing. If your agent behaves differently every run, check temperature first.
- **Embedding dimension mismatch.** You build a vector store with `text-embedding-3-small` (1536 dims) and later query it with `text-embedding-3-large` (3072 dims), or with `dimensions=512` after indexing at 1536. Every vector in a store must have the same dimension count. Decide the model and dimensions once, before indexing, and do not change them mid-project.
- **Rate limits on big embedding jobs.** Embedding a large corpus hits requests-per-minute caps. Lower `chunk_size`, add retries, and run overnight jobs in smaller passes rather than one giant call.
- **Embedding API costs.** At $0.02 per 1M tokens, a million documents of a thousand tokens each costs about $20 to embed once, then re-embedding after every chunking experiment multiplies it. Embed a sample first, iterate on the sample, then embed the full corpus.
- **Ollama not running.** `ChatOllama` fails with a connection error if the Ollama server is not up (`ollama serve`) or the model is not pulled (`ollama pull llama3.1`). Check `ollama list` before debugging your code.
- **Sending EU personal data to a hosted model.** Legal rather than technical, but it is your failure mode if the product ships. Know where your provider processes data, or keep sensitive workloads local with Ollama.

## Check your understanding

**1. What do `temperature=0` and `temperature=0.8` each do, and which would you use for a tool-calling agent?**
`temperature` controls randomness: 0 is deterministic and focused, 0.8 is varied and creative. Use 0 for the tool-calling agent, because you want the same inputs to produce the same tool calls every time.

**2. You have `gpt = init_chat_model("openai:gpt-4o-mini", temperature=0)` and want to try Anthropic instead. What changes?**
One line: `init_chat_model("anthropic:claude-sonnet-4-5", temperature=0)` (plus `pip install langchain-anthropic` and the API key). The factory resolves the right class from the `"provider:model"` prefix, so the chain code does not change.

**3. Why is `embed_documents(texts)` better than looping `embed_query(text)` for ten thousand documents?**
`embed_documents` batches many texts into a few API requests, while a loop makes ten thousand requests. That is fewer rate-limit hits, lower latency, and the same token cost. `embed_query` exists for single search queries.

**4. You indexed a vector store with `text-embedding-3-small`, then queried it with a `text-embedding-3-large` embedder. What breaks and why?**
The query vectors are 3072 dimensions while the stored vectors are 1536. Similarity search needs matching dimensions, so the query fails or returns garbage. Pick the model and dimension count once, before indexing, and keep them fixed.

**5. Your agent handles EU customer support chats and you are unsure whether chat transcripts may leave the EU. Name two ways to handle it.**
Use a hosted provider with EU data residency on an eligible plan, or run the model locally with Ollama (for example `ChatOllama`) so the data never leaves your machine.

## Done

Back to the lesson: https://logicstack.org/courses/ai-agent-engineering

When the code in this notebook feels easy, mark the lesson complete and move to the next one.